# MatchFormer Epipolar Fine-tuning — Google Colab

This notebook fine-tunes MatchFormer with epipolar supervision baked in.  
Checkpoints are saved to **Google Drive** so training can be resumed if Colab disconnects.

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Upload your ScanNet data to Google Drive (see Cell 3)
3. Run cells **top to bottom** — on resume, skip to Cell 6

## Cell 1: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints will be saved here — survives Colab restarts
DRIVE_CKPT_DIR = '/content/drive/MyDrive/final_proj/matchformer_checkpoints'

import os
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir: /content/drive/MyDrive/final_proj/matchformer_checkpoints


## Cell 2: Clone Repo & Install Dependencies

In [14]:
import os

REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'  # your repo
REPO_DIR = '/content/final_proj'

if not os.path.exists(REPO_DIR):
    print('Repo not cloned — cloning')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git pull

%cd /content/final_proj/MatchFormer

Repo already cloned — pulling latest
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 11 (delta 9), reused 11 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 1.22 KiB | 416.00 KiB/s, done.
From https://github.com/sid-raj-uc/match-former
   7df949c..b0c501f  main       -> origin/main
Updating 7df949c..b0c501f
Fast-forward
 MatchFormer/model/datasets/scannet_simple.py | 27 ++++++++++++++++-----------
 MatchFormer/model/lightning_loftr.py         |  6 +++---
 2 files changed, 19 insertions(+), 14 deletions(-)
/content/final_proj/MatchFormer


In [6]:
# Install dependencies
!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 149.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 93.1 MB/s eta 0:00:00
Dependencies installed.


## Cell 3: Upload ScanNet Data

Upload your exported ScanNet scene to Google Drive. Expected structure:
```
MyDrive/scannet_data/scene0000_00/exported/
    color/    *.jpg
    depth/    *.png
    pose/     *.txt
    intrinsic/intrinsic_depth.txt
```
Or zip and upload, then unzip below.

In [7]:
# Option A: Data already on Drive — just set the path
DATA_DIR = '/content/drive/MyDrive/final_proj/data/scans/scene0000_00/exported'

# Option B: Unzip from Drive
# ZIP_PATH = '/content/drive/MyDrive/scannet_scene0000_00.zip'
# !unzip -q {ZIP_PATH} -d /content/scannet_data/
# DATA_DIR = '/content/scannet_data/scene0000_00/exported'

# Verify
import os, glob
n_color = len(glob.glob(os.path.join(DATA_DIR, 'color', '*.jpg')))
n_depth = len(glob.glob(os.path.join(DATA_DIR, 'depth', '*.png')))
print(f'Found {n_color} color frames, {n_depth} depth frames')

Found 5578 color frames, 5578 depth frames


## Cell 4: Download Pretrained Weights

In [8]:
# Check if pretrained weight already exists (from a previous run or Drive)
WEIGHTS_PATH = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/final_proj/MatchFormer/model/weights/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(DRIVE_WEIGHTS) and not os.path.exists(WEIGHTS_PATH):
    import shutil
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
elif os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
else:
    print('⚠️  Please upload indoor-lite-LA.ckpt to Drive at:', DRIVE_WEIGHTS)
    print('   or place it at:', WEIGHTS_PATH)

Copied weights from Drive


## Cell 5: Verify GPU & Run Sanity Check
Run this once to confirm the pipeline works before doing the full training.

In [9]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"}')
print(f'CUDA available: {torch.cuda.is_available()}')

GPU: Tesla T4
CUDA available: True


In [12]:
# Sanity check: 5 pairs, 50 steps — should finish in ~2 min on T4
!python train_finetune.py \
    --overfit \
    --data_dir {DATA_DIR} \
    --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
    --steps 50

print('Sanity check complete — loss should be decreasing!')

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[Auto-resume] No checkpoint found — starting from scratch.
Dataset: 5 pairs
2026-03-16 02:03:10.059 | INFO     | model.lightning_loftr:__init__:48 - Load 'model/weights/indoor-lite-LA.ckpt' as pretrained checkpoint
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
2026-03-16 02:03:11.166731: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Un

## Cell 6: Full Fine-tuning

**Auto-resume is enabled** — if Colab disconnects, just re-run from Cell 1 and then run this cell again. It will automatically pick up from the last saved checkpoint in your Drive.

**Key hyperparameters:**
- `--steps 10000` — total training steps (~2-3 hours on T4)
- `--tau 10.0` — epipolar constraint strength
- `--save_every 200` — save checkpoint to Drive every 200 steps
- `--batch 4` — batch size (reduce to 2 if OOM)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!python train_finetune.py \
    --data_dir {DATA_DIR} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 10000 \
    --tau 10.0 \
    --batch 2 \
    --workers 2 \
    --lr 1e-4 \
    --save_every 200

# To resume manually from a specific checkpoint:
# !python train_finetune.py \
#     --data_dir {DATA_DIR} \
#     --checkpoint_dir {DRIVE_CKPT_DIR} \
#     --resume {DRIVE_CKPT_DIR}/last.ckpt \
#     --steps 10000 \
#     --tau 10.0 --batch 4 --save_every 200

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[Auto-resume] No checkpoint found — starting from scratch.
Dataset: 5558 pairs
2026-03-16 02:25:08.732 | INFO     | model.lightning_loftr:__init__:48 - Load 'model/weights/indoor-lite-LA.ckpt' as pretrained checkpoint
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
2026-03-16 02:25:09.694552: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register c

## Cell 7: Run Benchmark After Training

In [ ]:
# After training finishes, copy the best checkpoint locally and run benchmark
import shutil

TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'
shutil.copy(TRAINED_CKPT, 'model/weights/epipolar-finetuned.ckpt')
print('Checkpoint copied.')

# Run benchmark with finetuned model
!python run_benchmark.py \
    --num_pairs 100 \
    --ckpt model/weights/epipolar-finetuned.ckpt \
    2>&1 | tee benchmark_finetuned_log.txt

# Copy results back to Drive
shutil.copy('benchmark_finetuned_log.txt', f'{DRIVE_CKPT_DIR}/benchmark_finetuned_log.txt')
print('Benchmark results saved to Drive!')